# 순위형 선호모델 실험

- 실험명: rank_ordered_and_ranker_models
- 입력변수: 기존 최종 후보 변수셋으로 고정함
- 실험모델: Rank-Ordered Logit, LightGBM LambdaRank, XGBoost rank:pairwise, XGBoost rank:ndcg, CatBoost ranking
- 목적: 1~3순위 응답의 순서 정보를 직접 활용하는 모델을 비교함

## 1. 패키지 및 경로

- 국민여가활동조사 기반 선호도 데이터를 사용함.
- 매핑표를 이용해 여가활동 코드를 문화누리 중분류로 변환함.
- 별도 CSV 산출물은 생성하지 않음.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import logsumexp, softmax

from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, log_loss, top_k_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRanker, Pool

warnings.filterwarnings("ignore")

BASE_PATH = Path.cwd()
while BASE_PATH.name != "oracle_mnc_project" and BASE_PATH.parent != BASE_PATH:
    BASE_PATH = BASE_PATH.parent

RAW_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "source" / "leisure_activity_survey_2021_2025_selected_columns_enriched.csv"
MAPPING_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "processed" / "satisfaction" / "ml_activity_category_mapping.csv"

print("RAW_PATH 존재:", RAW_PATH.exists())
print("MAPPING_PATH 존재:", MAPPING_PATH.exists())

## 2. 선호도 순위 데이터 생성

- 향후 희망 여가활동 1~3순위를 사용함.
- 교통수단·여행사·음악·체육용품은 제외함.
- 순위형 모델은 순서 자체를 사용하므로 1.5/1.25/1.0 가중치는 사용하지 않음.
- 조사 설계 가중치인 최종가중치는 평가와 Rank-Ordered Logit 추정에 반영함.

In [ ]:
raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig")
mapping = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")
mapping.columns = ["activity_code", "activity_name", "category", "use_target"]

# 기준 중위소득 100%, 원/월
median_income_100 = {
    2024: {
        1: 2228445, 2: 3682609, 3: 4714657, 4: 5729913,
        5: 6695735, 6: 7618369, 7: 8514994,
    },
    2025: {
        1: 2392013, 2: 3932658, 3: 5025353, 4: 6097773,
        5: 7108192, 6: 8064805, 7: 8988428,
    },
}

income_mid_10k = {
    1: 50,
    2: 150,
    3: 250,
    4: 350,
    5: 450,
    6: 550,
    7: 650,
}

def household_size_cap(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x < 1:
        return np.nan
    return min(x, 7)

def median_ratio(row):
    year = int(row["조사년도"])
    size = household_size_cap(row["동거가구원수_생성"])
    income_code = row["가구소득_생성"]

    if pd.isna(size) or pd.isna(income_code):
        return np.nan

    income_won = income_mid_10k.get(int(income_code), np.nan) * 10000
    median_won = median_income_100.get(year, {}).get(size, np.nan)

    if pd.isna(income_won) or pd.isna(median_won) or median_won == 0:
        return np.nan

    return income_won / median_won

raw["중위소득비율_근사"] = raw.apply(median_ratio, axis=1)
raw["소득구간_다층"] = pd.cut(
    raw["중위소득비율_근사"],
    bins=[-np.inf, 0.5, 1.0, 1.5, np.inf],
    labels=["50이하", "50_100", "100_150", "150초과"],
).astype(str)
raw.loc[raw["중위소득비율_근사"].isna(), "소득구간_다층"] = "unknown"

excluded_categories = ["분류범위외", "교통수단", "여행사", "음악", "체육용품"]
mapping["use_target_final"] = (
    mapping["use_target"].astype(bool)
    & ~mapping["category"].isin(excluded_categories)
)

rank_cols = {
    1: "향후 희망하는 여가활동 1순위",
    2: "향후 희망하는 여가활동 2순위",
    3: "향후 희망하는 여가활동 3순위",
}

feature_cols = [
    "성별",
    "연령",
    "조사년도",
    "성별_연령",
    "소득구간_다층",
    "장애여부",
    "17개 시도",
]

preference_raw = raw.loc[
    raw["조사년도"].isin([2024, 2025]),
    [
        "응답자_ID", "최종가중치", "성별", "연령", "조사년도",
        "소득구간_다층", "장애여부", "17개 시도",
    ] + list(rank_cols.values())
].copy()

preference_raw["성별_연령"] = (
    preference_raw["성별"].astype("Int64").astype(str)
    + "_"
    + preference_raw["연령"].astype("Int64").astype(str)
)

for col in feature_cols:
    preference_raw[col] = preference_raw[col].astype("string").fillna("unknown")

code_to_category = mapping.set_index("activity_code")["category"].to_dict()
code_to_use = mapping.set_index("activity_code")["use_target_final"].to_dict()

wide_records = []
long_records = []

for _, row in preference_raw.iterrows():
    valid_rows = []
    seen_categories = set()

    for rank_no, col in rank_cols.items():
        activity_code = row[col]
        if pd.isna(activity_code):
            continue

        activity_code = int(activity_code)
        category = code_to_category.get(activity_code)
        use_target = bool(code_to_use.get(activity_code, False))

        if not use_target:
            continue
        if category in seen_categories:
            continue

        seen_categories.add(category)
        valid_rows.append({
            "rank_no": rank_no,
            "target_category": category,
            "rank_label": 4 - rank_no,
        })

    wide_row = {
        "응답자_ID": row["응답자_ID"],
        "성별": row["성별"],
        "연령": row["연령"],
        "조사년도": row["조사년도"],
        "성별_연령": row["성별_연령"],
        "소득구간_다층": row["소득구간_다층"],
        "장애여부": row["장애여부"],
        "17개 시도": row["17개 시도"],
        "최종가중치": row["최종가중치"],
        "선호_유효순위수": len(valid_rows),
    }

    for i, valid in enumerate(valid_rows, start=1):
        wide_row[f"선호_유효중분류_{i}순위"] = valid["target_category"]
        long_records.append({
            "응답자_ID": row["응답자_ID"],
            "rank_no": valid["rank_no"],
            "target_category": valid["target_category"],
            "rank_label": valid["rank_label"],
        })

    wide_records.append(wide_row)

rank_base = pd.DataFrame(wide_records)
rank_long = pd.DataFrame(long_records)

rank_base_valid = rank_base.loc[rank_base["선호_유효순위수"] > 0].copy()
classes = np.array(sorted(rank_long["target_category"].unique()))
class_to_idx = {c: i for i, c in enumerate(classes)}

primary_target = (
    rank_long.sort_values(["응답자_ID", "rank_no"])
    .groupby("응답자_ID", as_index=False)["target_category"]
    .first()
    .rename(columns={"target_category": "primary_target"})
)

rank_base_valid = rank_base_valid.merge(primary_target, on="응답자_ID", how="left")

print("응답자 테이블:", rank_base.shape)
print("유효 응답자:", rank_base_valid.shape)
print("순위 long 테이블:", rank_long.shape)
print("분류:", classes)
print("유효 순위 수")
print(rank_base["선호_유효순위수"].value_counts().sort_index().to_string())

## 3. Train/Valid/Test 분할

- 응답자 기준으로 분리함.
- train/test = 8:2
- train 내부 train/valid = 8:2
- 조사년도와 1순위 타깃을 함께 고려해 stratify함.

In [ ]:
def choose_strata(df, min_count=2):
    year_target = df["조사년도"].astype(str) + "_" + df["primary_target"].astype(str)
    if year_target.value_counts().min() >= min_count:
        return year_target

    target_only = df["primary_target"].astype(str)
    if target_only.value_counts().min() >= min_count:
        return target_only

    return None

train_valid_base, test_base = train_test_split(
    rank_base_valid,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=choose_strata(rank_base_valid, 2),
)

train_base, valid_base = train_test_split(
    train_valid_base,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=choose_strata(train_valid_base, 2),
)

split_summary = pd.DataFrame({
    "dataset": ["train", "valid", "test"],
    "respondents": [
        train_base["응답자_ID"].nunique(),
        valid_base["응답자_ID"].nunique(),
        test_base["응답자_ID"].nunique(),
    ],
})
split_summary["share"] = split_summary["respondents"] / rank_base_valid["응답자_ID"].nunique()
display(split_summary)

## 4. 순위모델 학습 데이터 생성

- Rank-Ordered Logit은 응답자의 실제 1~3순위 선택 순서를 사용함.
- 랭킹 모델은 응답자별 후보 중분류 전체를 만들고, 1순위=3점, 2순위=2점, 3순위=1점, 미등장=0점으로 학습함.
- 후보 중분류 전체를 같은 group으로 묶어 모델에 전달함.

In [ ]:
def make_rank_sequence_dict(long_df):
    seq = {}
    for rid, grp in long_df.sort_values(["응답자_ID", "rank_no"]).groupby("응답자_ID"):
        seq[rid] = grp["target_category"].tolist()
    return seq

rank_sequences = make_rank_sequence_dict(rank_long)

def make_ranker_frame(base_df):
    records = []
    for _, row in base_df.sort_values("응답자_ID").iterrows():
        rid = row["응답자_ID"]
        seq = rank_sequences.get(rid, [])
        label_map = {cat: 3 - i for i, cat in enumerate(seq[:3])}

        for cat in classes:
            rec = {
                "응답자_ID": rid,
                "alternative_category": cat,
                "rank_label": label_map.get(cat, 0),
                "group_weight": row["최종가중치"],
            }
            for col in feature_cols:
                rec[col] = row[col]
            records.append(rec)

    return pd.DataFrame(records)

train_ranker = make_ranker_frame(train_base)
valid_ranker = make_ranker_frame(valid_base)
test_ranker = make_ranker_frame(test_base)

ranker_cat_cols = feature_cols + ["alternative_category"]

encoder = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ranker_cat_cols)],
    remainder="drop",
)

encoder.fit(train_ranker[ranker_cat_cols].astype(str))

def transform_ranker(df):
    x = encoder.transform(df[ranker_cat_cols].astype(str))
    y = df["rank_label"].to_numpy()
    group = df.groupby("응답자_ID", sort=False).size().to_numpy()
    group_weight = df.groupby("응답자_ID", sort=False)["group_weight"].first().to_numpy()
    row_weight = df["group_weight"].to_numpy()
    return x, y, group, group_weight, row_weight

x_train, y_train, group_train, group_weight_train, row_weight_train = transform_ranker(train_ranker)
x_valid, y_valid, group_valid, group_weight_valid, row_weight_valid = transform_ranker(valid_ranker)
x_test, y_test, group_test, group_weight_test, row_weight_test = transform_ranker(test_ranker)

print("train ranker:", train_ranker.shape, x_train.shape)
print("valid ranker:", valid_ranker.shape, x_valid.shape)
print("test ranker:", test_ranker.shape, x_test.shape)
print("group 크기:", np.unique(group_train))

## 5. 평가 함수

- LogLoss: 1순위 중분류에 대한 확률 오차임.
- Top1: 예측 1위가 실제 1순위와 같은지 봄.
- Top3: 실제 1순위가 예측 상위 3개 안에 있는지 봄.
- NDCG@3: 1~3순위 전체 순서 품질을 봄.
- PL_NLL: Plackett-Luce 방식으로 실제 순위열의 음의 로그가능도를 계산함.

In [ ]:
def scores_to_matrix(base_df, score_vector):
    n_group = len(base_df)
    return np.asarray(score_vector).reshape(n_group, len(classes))

def relevance_matrix(base_df):
    rel = np.zeros((len(base_df), len(classes)))
    for i, rid in enumerate(base_df["응답자_ID"]):
        seq = rank_sequences.get(rid, [])
        for rank_i, cat in enumerate(seq[:3]):
            rel[i, class_to_idx[cat]] = 3 - rank_i
    return rel

def primary_target_index(base_df):
    return np.array([class_to_idx[x] for x in base_df["primary_target"]])

def weighted_ndcg_at_k(rel, scores, weights, k=3):
    order = np.argsort(-scores, axis=1)[:, :k]
    gains = np.take_along_axis(rel, order, axis=1)
    discounts = 1 / np.log2(np.arange(2, k + 2))
    dcg = (gains * discounts).sum(axis=1)

    ideal_order = np.argsort(-rel, axis=1)[:, :k]
    ideal_gains = np.take_along_axis(rel, ideal_order, axis=1)
    idcg = (ideal_gains * discounts).sum(axis=1)

    ndcg = np.divide(dcg, idcg, out=np.zeros_like(dcg), where=idcg > 0)
    return np.average(ndcg, weights=weights)

def pl_nll_from_scores(base_df, scores, weights):
    nll = []
    for i, rid in enumerate(base_df["응답자_ID"]):
        seq = rank_sequences.get(rid, [])
        remaining = list(range(len(classes)))
        total = 0.0

        for cat in seq:
            chosen = class_to_idx[cat]
            if chosen not in remaining:
                continue
            remain_scores = scores[i, remaining]
            chosen_pos = remaining.index(chosen)
            total += -(remain_scores[chosen_pos] - logsumexp(remain_scores))
            remaining.remove(chosen)

        nll.append(total)

    return np.average(np.array(nll), weights=weights)

def evaluate_scores(experiment, dataset_name, base_df, scores):
    weights = base_df["최종가중치"].to_numpy()
    target_idx = primary_target_index(base_df)
    prob = softmax(scores, axis=1)
    pred_idx = np.argmax(prob, axis=1)

    rel = relevance_matrix(base_df)

    return {
        "experiment": experiment,
        "dataset": dataset_name,
        "LogLoss": log_loss(
            target_idx,
            prob,
            labels=np.arange(len(classes)),
            sample_weight=weights,
        ),
        "Top1_Accuracy": accuracy_score(target_idx, pred_idx, sample_weight=weights),
        "Top3_HitRate": top_k_accuracy_score(
            target_idx,
            prob,
            k=3,
            labels=np.arange(len(classes)),
            sample_weight=weights,
        ),
        "NDCG@3": weighted_ndcg_at_k(rel, scores, weights, k=3),
        "PL_NLL": pl_nll_from_scores(base_df, scores, weights),
    }

## 6. Rank-Ordered Logit

- 응답자가 남은 선택지 중 하나를 순서대로 선택한다고 가정함.
- 각 중분류별 효용은 입력변수와 중분류의 상호작용으로 추정함.
- 기준 중분류 하나는 계수를 0으로 고정해 식별함.

In [ ]:
rol_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
rol_encoder.fit(train_base[feature_cols].astype(str))

def rol_x(base_df):
    x = rol_encoder.transform(base_df[feature_cols].astype(str))
    intercept = np.ones((x.shape[0], 1))
    return np.hstack([intercept, x])

x_train_rol = rol_x(train_base)
x_valid_rol = rol_x(valid_base)
x_test_rol = rol_x(test_base)

n_class = len(classes)
n_param_class = n_class - 1
n_feature = x_train_rol.shape[1]

def unpack_theta(theta):
    beta = theta.reshape(n_param_class, n_feature)
    beta_full = np.vstack([np.zeros((1, n_feature)), beta])
    return beta_full

def make_rol_aggregated(base_df, x_mat):
    temp = base_df[["응답자_ID", "최종가중치"]].copy()
    temp["seq_key"] = temp["응답자_ID"].map(
        lambda rid: "|".join(rank_sequences.get(rid, []))
    )
    temp["x_key"] = list(map(tuple, x_mat))

    grouped = (
        temp.groupby(["seq_key", "x_key"], as_index=False)["최종가중치"]
        .sum()
        .rename(columns={"최종가중치": "group_weight"})
    )

    x_agg = np.array(grouped["x_key"].tolist(), dtype=float)
    seq_agg = grouped["seq_key"].str.split("|").tolist()
    w_agg = grouped["group_weight"].to_numpy()

    return x_agg, seq_agg, w_agg

x_train_rol_agg, seq_train_rol_agg, w_train_rol_agg = make_rol_aggregated(
    train_base,
    x_train_rol,
)

print("ROL 원 응답자 수:", len(train_base))
print("ROL 집계 패턴 수:", len(seq_train_rol_agg))

def rol_objective(theta, x_mat, seq_list, weight_arr):
    beta_full = unpack_theta(theta)
    scores = x_mat @ beta_full.T
    grad_full = np.zeros_like(beta_full)
    loss = 0.0

    for i, seq in enumerate(seq_list):
        weight = weight_arr[i]
        remaining = list(range(n_class))
        xi = x_mat[i]

        for cat in seq:
            chosen = class_to_idx[cat]
            if chosen not in remaining:
                continue

            remain_scores = scores[i, remaining]
            chosen_pos = remaining.index(chosen)
            probs = softmax(remain_scores)

            loss += weight * (-(remain_scores[chosen_pos] - logsumexp(remain_scores)))

            for pos, alt_idx in enumerate(remaining):
                grad_full[alt_idx] += weight * probs[pos] * xi
            grad_full[chosen] -= weight * xi

            remaining.remove(chosen)

    grad = grad_full[1:].reshape(-1)
    weight_sum = weight_arr.sum()
    return loss / weight_sum, grad / weight_sum

theta0 = np.zeros(n_param_class * n_feature)

rol_result = minimize(
    fun=lambda th: rol_objective(th, x_train_rol_agg, seq_train_rol_agg, w_train_rol_agg),
    x0=theta0,
    method="L-BFGS-B",
    jac=True,
    options={"maxiter": 70, "disp": False, "maxls": 20},
)

print("수렴 여부:", rol_result.success)
print("반복 수:", rol_result.nit)
print("최종 loss:", rol_result.fun)

rol_beta = unpack_theta(rol_result.x)

rol_scores = {
    "train": x_train_rol @ rol_beta.T,
    "valid": x_valid_rol @ rol_beta.T,
    "test": x_test_rol @ rol_beta.T,
}

## 7. LightGBM·XGBoost·CatBoost 랭킹 모델

- LightGBM은 LambdaRank 목적함수를 사용함.
- XGBoost는 rank:pairwise와 rank:ndcg를 각각 실험함.
- CatBoost는 YetiRank를 사용함.

In [ ]:
ranker_scores = {}

print("LightGBM LambdaRank 학습")
lgb_ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=80,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=20,
    random_state=42,
    verbosity=-1,
)
lgb_ranker.fit(
    x_train,
    y_train,
    group=group_train,
    sample_weight=row_weight_train,
)
ranker_scores["lightgbm_lambdarank"] = {
    "train": scores_to_matrix(train_base, lgb_ranker.predict(x_train)),
    "valid": scores_to_matrix(valid_base, lgb_ranker.predict(x_valid)),
    "test": scores_to_matrix(test_base, lgb_ranker.predict(x_test)),
}

for objective in ["rank:pairwise", "rank:ndcg"]:
    print("XGBoost", objective, "학습")
    xgb_ranker = xgb.XGBRanker(
        objective=objective,
        n_estimators=80,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        tree_method="hist",
        n_jobs=2,
    )
    try:
        xgb_ranker.fit(
            x_train,
            y_train,
            group=group_train,
            sample_weight=group_weight_train,
            verbose=False,
        )
        print("XGBoost group weight 적용")
    except Exception as e:
        print("XGBoost group weight 미적용:", e)
        xgb_ranker.fit(
            x_train,
            y_train,
            group=group_train,
            verbose=False,
        )

    key = "xgboost_" + objective.replace(":", "_")
    ranker_scores[key] = {
        "train": scores_to_matrix(train_base, xgb_ranker.predict(x_train)),
        "valid": scores_to_matrix(valid_base, xgb_ranker.predict(x_valid)),
        "test": scores_to_matrix(test_base, xgb_ranker.predict(x_test)),
    }

print("CatBoost YetiRank 학습")
cat_train_pool = Pool(
    x_train,
    label=y_train,
    group_id=train_ranker["응답자_ID"].to_numpy(),
    weight=row_weight_train,
)
cat_ranker = CatBoostRanker(
    loss_function="YetiRank",
    iterations=80,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)
cat_ranker.fit(cat_train_pool)

ranker_scores["catboost_yetirank"] = {
    "train": scores_to_matrix(train_base, cat_ranker.predict(x_train)),
    "valid": scores_to_matrix(valid_base, cat_ranker.predict(x_valid)),
    "test": scores_to_matrix(test_base, cat_ranker.predict(x_test)),
}

## 8. 성능 비교

- 모든 모델을 동일한 test set에서 비교함.
- H3SFCA 수요확률에 활용하려면 LogLoss와 PL_NLL을 우선 확인함.
- 순위 품질 자체는 NDCG@3도 함께 확인함.

In [ ]:
performance_rows = []

for dataset_name, base_df in [
    ("train", train_base),
    ("valid", valid_base),
    ("test", test_base),
]:
    performance_rows.append(
        evaluate_scores("rank_ordered_logit", dataset_name, base_df, rol_scores[dataset_name])
    )

for exp_name, score_dict in ranker_scores.items():
    for dataset_name, base_df in [
        ("train", train_base),
        ("valid", valid_base),
        ("test", test_base),
    ]:
        performance_rows.append(
            evaluate_scores(exp_name, dataset_name, base_df, score_dict[dataset_name])
        )

performance = pd.DataFrame(performance_rows)

print("Holdout Test")
test_performance = performance[performance["dataset"] == "test"].sort_values("LogLoss")
display(test_performance)

print("Valid")
display(performance[performance["dataset"] == "valid"].sort_values("LogLoss"))

## 9. 실험 결과 요약

- 입력변수는 `성별`, `연령`, `조사년도`, `성별_연령`, `소득구간_다층`, `장애여부`, `17개 시도`로 고정함.
- Rank-Ordered Logit은 실제 1~3순위 선택 순서를 직접 사용함.
- 랭킹 모델은 응답자별 후보 중분류 전체를 만들고 1순위=3, 2순위=2, 3순위=1, 미등장=0으로 학습함.
- LightGBM은 LambdaRank, XGBoost는 rank:pairwise/rank:ndcg, CatBoost는 YetiRank를 사용함.

### Holdout Test 결과

```text
           experiment dataset  LogLoss  Top1_Accuracy  Top3_HitRate   NDCG@3   PL_NLL
   rank_ordered_logit    test 1.716713       0.314819      0.727521 0.608621 2.776028
    catboost_yetirank    test 1.819240       0.294111      0.701841 0.585644 2.945094
xgboost_rank_pairwise    test 1.819532       0.295819      0.706652 0.586510 2.940546
    xgboost_rank_ndcg    test 1.841222       0.274298      0.714483 0.583770 2.968746
  lightgbm_lambdarank    test 1.841785       0.278273      0.699891 0.575143 2.982397
```